In [1]:
!pip install duckduckgo_search

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.9 MB 3.9 MB/s eta 0:00:01
   ------------- -------------------------- 1.3/3.9 MB 3.9 MB/s eta 0:00:01
   --------------------- ------------------ 2.1/3.9 MB 3.9 MB/s eta 0:00:01
   ----------------------------- ---------- 2.9/3.9 MB 3.9 MB/s eta 0:00:01
   ------------------------------------- -- 3.7/3.9 MB 3.8 MB/s eta 0:00:01
   ---------------------------------------- 3.9/3.9 MB 3.9 MB/s  0:00:01

   -------------------- ------------------- 1/2 [duckduckgo_search]
   ---------------------------------------- 2/2 [duckduckgo_search]




[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Cell 1: Install correct package
!pip install ddgs

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\seif almaz\AppData\Roaming\Python\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
C:\Users\seif almaz\AppData\Roaming\Python\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
C:\Users\seif almaz\AppData\Roaming\Python\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  res = process_handler(cmd, _system_body)


In [5]:

from ddgs import DDGS
import requests, os, time, random
from PIL import Image
from io import BytesIO
from tqdm import tqdm

BASE_DIR = "../data/raw"

CLASS_QUERIES = {
    "Plastic":  ["plastic waste bottle", "plastic bag garbage", "plastic trash recycling", "plastic pollution waste"],
    "Paper":    ["paper waste cardboard", "newspaper trash", "paper recycling bin", "cardboard waste garbage"],
    "Metal":    ["metal can waste", "aluminum tin garbage", "scrap metal recycling", "steel can trash"],
    "Glass":    ["glass bottle waste", "glass jar recycling", "broken glass garbage", "glass waste disposal"],
    "Organic":  ["food waste compost", "fruit scraps organic", "kitchen waste vegetable", "organic garbage bin"],
}

TARGET = 1000

def download_image(url, path):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        r = requests.get(url, timeout=6, headers=headers)
        img = Image.open(BytesIO(r.content)).convert("RGB")
        if min(img.size) >= 100:
            img.save(path)
            return True
    except:
        pass
    return False

def scrape_ddg(query, max_results=150):
    """Fetch image URLs with retry on rate limit."""
    for attempt in range(3):
        try:
            with DDGS() as ddgs:
                results = list(ddgs.images(query, max_results=max_results))
            return results
        except Exception as e:
            wait = 30 * (attempt + 1)  # 30s, 60s, 90s
            print(f"    ⚠️  Rate limited on '{query}' — waiting {wait}s...")
            time.sleep(wait)
    return []  # give up after 3 attempts

for class_name, queries in CLASS_QUERIES.items():
    save_dir = os.path.join(BASE_DIR, class_name)
    os.makedirs(save_dir, exist_ok=True)

    existing = len([f for f in os.listdir(save_dir)
                    if f.lower().endswith(('.jpg','.jpeg','.png'))])

    if existing >= TARGET:
        print(f"✅ {class_name} already has {existing} — skipping")
        continue

    needed = TARGET - existing
    print(f"\n📥 {class_name} — have {existing}, need {needed} more")

    downloaded = 0
    idx = existing

    for query in queries:
        if downloaded >= needed:
            break

        print(f"  🔍 Query: '{query}'")
        results = scrape_ddg(query, max_results=150)

        if not results:
            print(f"  ❌ No results for '{query}', skipping")
            continue

        for r in tqdm(results, desc=f"  Downloading"):
            if downloaded >= needed:
                break
            path = os.path.join(save_dir, f"ddg_{idx:06d}.jpg")
            if download_image(r["image"], path):
                downloaded += 1
                idx += 1

        # Polite delay between queries to avoid rate limiting
        delay = random.uniform(8, 15)
        print(f"  ⏳ Waiting {delay:.0f}s before next query...")
        time.sleep(delay)

    total = len([f for f in os.listdir(save_dir)
                 if f.lower().endswith(('.jpg','.jpeg','.png'))])
    print(f"  → {class_name} total: {total}")

print("\n✅ DDG scraping done!")


📥 Plastic — have 458, need 542 more
  🔍 Query: 'plastic waste bottle'


  Downloading: 100%|███████████████████████████████████████████████████████████████████| 35/35 [00:43<00:00,  1.26s/it]


  ⏳ Waiting 13s before next query...
  🔍 Query: 'plastic bag garbage'


  Downloading: 100%|███████████████████████████████████████████████████████████████████| 35/35 [00:29<00:00,  1.20it/s]


  ⏳ Waiting 14s before next query...
  🔍 Query: 'plastic trash recycling'


...

  ⏳ Waiting 9s before next query...
  🔍 Query: 'kitchen waste vegetable'


  Downloading: 100%|███████████████████████████████████████████████████████████████████| 35/35 [00:29<00:00,  1.18it/s]


  ⏳ Waiting 9s before next query...
  🔍 Query: 'organic garbage bin'


  Downloading: 100%|███████████████████████████████████████████████████████████████████| 49/49 [00:46<00:00,  1.06it/s]


  ⏳ Waiting 9s before next query...
  → Organic total: 607

✅ DDG scraping done!
